In [12]:
# pip freeze > requirements.txt
# pip install -r requirements.txt
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity
import nltk

# Download stopwords
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mehmet\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
# Load the filtered dataset
file_path = "filtered_file_2.xlsx"  # Replace with your actual file path
df = pd.read_excel(file_path)

# Display the first few rows
df.head()


,Article Title,Abstract,Researcher Ids,ORCIDs,DOI,WoS Categories,Research Areas,IDS Number,UT (Unique WOS ID),Publication Year
0,TRANSIENT-RESPONSE OF LATENT-HEAT STORAGE IN G...,A latent heat storage system with two differen...,"Cengel, Yunus/D-4105-2015",NaN,10.1016/0038-092X(86)90045-9,Energy & Fuels,Energy & Fuels,E4303,WOS:A1986E430300004,1986
1,THE EFFECT OF TESTICULAR TORSION ON THE CONTRA...,In an attempt to investigate the effect of tes...,"Canda, M.Serefittin/K-4298-2019",NaN,10.1111/j.1464-410X.1988.tb04269.x,Urology & Nephrology,Urology & Nephrology,P2197,WOS:A1988P219700017,1988
2,RADIOACTIVE POLLUTION OF TURKISH BIOTAS ONE YE...,Long-lived fission radionuclides spread out af...,"AKCAY, HUSEYIN/ABI-8239-2020",NaN,10.1007/BF02166951,"Chemistry, Analytical; Chemistry, Inorganic & ...",Chemistry; Nuclear Science & Technology,R0473,WOS:A1988R047300004,1988
3,PROSTHESES DESIGNED FOR VERTEBRAL BODY REPLACE...,Prostheses are proposed to restore the spinal ...,"Alıcı Davutoğlu, Ebru/AIE-5610-2022",NaN,10.1016/0021-9290(90)90027-Z,"Biophysics; Engineering, Biomedical",Biophysics; Engineering,DP196,WOS:A1990DP19600007,1990
4,INTERANNUAL VARIABILITY IN PARTICLE-FLUX IN TH...,Vertical particle flux was measured at two sit...,"Ittekkot, Venugopalan/AAC-3437-2022; Kempe, St...","Ittekkot, Venugopalan/0000-0003-2667-5068",10.1016/0198-0149(90)90103-3,Oceanography,Oceanography,DQ544,WOS:A1990DQ54400002,1990


In [4]:
# Define a function to preprocess text
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'[^\w\s]', '', text.lower())  # Remove punctuation and convert to lowercase
    text = ' '.join(word for word in text.split() if word not in stopwords.words('english'))  # Remove stopwords
    return text

# Apply text preprocessing
df['Abstract_clean'] = df['Abstract'].apply(preprocess_text)
df['Research_Areas_clean'] = df['Research Areas'].apply(preprocess_text)

# Combine text columns for analysis
df['combined_text'] = df['Abstract_clean'] + ' ' + df['Research_Areas_clean']

# Display cleaned data
df[['combined_text']].head()


,combined_text
0,latent heat storage system two different stack...
1,attempt investigate effect testicular torsion ...
2,longlived fission radionuclides spread chernob...
3,prostheses proposed restore spinal stability p...
4,vertical particle flux measured two sites sout...


In [5]:
# Function to split researchers into individual rows
def split_researchers(row):
    if not isinstance(row['Researcher Ids'], str):
        return pd.DataFrame()  # Skip invalid rows
    researchers = row['Researcher Ids'].split(';')  # Split by semicolons
    data = []
    for researcher in researchers:
        data.append({
            'Researcher': researcher.strip(),
            'combined_text': row['combined_text'],
            'Publication Year': row['Publication Year']
        })
    return pd.DataFrame(data)

# Flatten the dataset by splitting researchers
flattened_df = pd.concat(df.apply(split_researchers, axis=1).to_list(), ignore_index=True)

# Display the flattened DataFrame
flattened_df.head()


,Researcher,combined_text,Publication Year
0,"Cengel, Yunus/D-4105-2015",latent heat storage system two different stack...,1986
1,"Canda, M.Serefittin/K-4298-2019",attempt investigate effect testicular torsion ...,1988
2,"AKCAY, HUSEYIN/ABI-8239-2020",longlived fission radionuclides spread chernob...,1988
3,"Alıcı Davutoğlu, Ebru/AIE-5610-2022",prostheses proposed restore spinal stability p...,1990
4,"Ittekkot, Venugopalan/AAC-3437-2022",vertical particle flux measured two sites sout...,1990


In [6]:
# Initialize CountVectorizer
count_vectorizer = CountVectorizer(max_features=1000, stop_words='english')

# Fit and transform the combined text data
count_matrix = count_vectorizer.fit_transform(flattened_df['combined_text'])

# Display the shape of the document-term matrix
print("Document-term matrix shape:", count_matrix.shape)


Document-term matrix shape: (68569, 1000)


In [7]:
# Initialize LDA model
lda = LatentDirichletAllocation(
    n_components=20,        # Number of topics
    max_iter=15,            # Max iterations for convergence
    learning_method='online',  # For large datasets
    learning_decay=0.7,     # Balances new vs old data
    learning_offset=50.0,   # Stabilizes early iterations
    random_state=42         # Reproducibility
)

# Fit the LDA model
lda_matrix = lda.fit_transform(count_matrix)

# Add topic distributions to the DataFrame
flattened_df['topic_distribution'] = list(lda_matrix)

# Display topic distributions
flattened_df.head()


,Researcher,combined_text,Publication Year,topic_distribution
0,"Cengel, Yunus/D-4105-2015",latent heat storage system two different stack...,1986,"[0.0025000001033412744, 0.0025000000314563684,..."
1,"Canda, M.Serefittin/K-4298-2019",attempt investigate effect testicular torsion ...,1988,"[0.001315789483604646, 0.0013157894736880392, ..."
2,"AKCAY, HUSEYIN/ABI-8239-2020",longlived fission radionuclides spread chernob...,1988,"[0.0019230769776929432, 0.0019230769700387584,..."
3,"Alıcı Davutoğlu, Ebru/AIE-5610-2022",prostheses proposed restore spinal stability p...,1990,"[0.16251610618930115, 0.0019230769302877072, 0..."
4,"Ittekkot, Venugopalan/AAC-3437-2022",vertical particle flux measured two sites sout...,1990,"[0.0021739130754467144, 0.002173913087122298, ..."


In [8]:
# Function to compute weighted aggregation
def weighted_aggregate(group, user_lda_vector):
    """
    Aggregates topic distributions for a researcher, weighted by article relevance,
    with improved handling of the number of articles.
    """
    # Compute relevance (cosine similarity) for each article
    relevance_scores = group['topic_distribution'].apply(
        lambda x: cosine_similarity([user_lda_vector], [x]).flatten()[0]
    )
    group['relevance'] = relevance_scores

    # Compute weighted average of topic distributions
    weighted_topics = np.average(
        np.vstack(group['topic_distribution']),
        axis=0,
        weights=relevance_scores
    )
    
    # Compute the number of articles
    num_articles = len(group)
    
    # Gaussian-like adjustment: peak impact at 15 articles
    optimal_point = 25
    adjustment_factor_part = max(0, 1 - ((num_articles - optimal_point) ** 2) / ((2 * optimal_point) ** 2))
    adjustment_factor = 1 + adjustment_factor_part * 0.3  # Minimal adjustment

    # Combine adjustment factor with average relevance
    adjusted_relevance = relevance_scores.mean() * adjustment_factor

    return pd.Series({
        'aggregated_topics': weighted_topics,   # Weighted topic distribution
        'num_articles': num_articles,          # Number of articles
        'adjusted_relevance': adjusted_relevance  # Adjusted relevance
    })


In [9]:
# Define the user's query
user_input = "cell cancer bio bioinformatics machine learning algorithm human life gene gen heredity"  # Replace with your query
user_input_clean = preprocess_text(user_input)

# Transform user query into the document-term matrix
user_count_vector = count_vectorizer.transform([user_input_clean])

# Transform user query into the LDA topic space
user_lda_vector = lda.transform(user_count_vector).flatten()

# Apply weighted aggregation for each researcher
grouped_df = flattened_df.groupby('Researcher').apply(lambda group: weighted_aggregate(group, user_lda_vector)).reset_index()

# Display grouped data
grouped_df.head()

#----------------------------------------------------------------------------------

# Compute similarity between user query and researchers
grouped_df['similarity'] = grouped_df['aggregated_topics'].apply(
    lambda x: cosine_similarity([user_lda_vector], [x]).flatten()[0]
)

# Combine similarity with adjusted relevance and article count
grouped_df['final_score'] = grouped_df['similarity'] * grouped_df['adjusted_relevance'] * np.log1p(grouped_df['num_articles'])

# Sort researchers by the final score
top_matches = grouped_df.sort_values(by='final_score', ascending=False).head(20)

# Display top matches
print("Top Recommended Collaborators:")
print(top_matches[['Researcher', 'final_score', 'num_articles', 'adjusted_relevance']])


C:\Users\mehmet\AppData\Local\Temp\ipykernel_12272\501850995.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_df = flattened_df.groupby('Researcher').apply(lambda group: weighted_aggregate(group, user_lda_vector)).reset_index()


Top Recommended Collaborators:
                                  Researcher  final_score  num_articles  \
12592   Pavlopoulou, Athanasia/ABB-3275-2021     2.993670            42   
6463   Georgakilas, Alexandros/AAG-9517-2020     2.814355            20   
8750           Karakulah, Gokhan/N-1342-2018     2.804907            49   
2147         Basbinar, Yasemin/AFW-2981-2022     2.743610            51   
3294       Calibasi Kocal, Gizem/G-1761-2014     2.576902            32   
7951              Işık, Zerrin/ABG-9392-2021     2.575826            22   
11488          Nasibov, Efendi/ISV-0173-2023     2.488510            46   
2251            Baykasoglu, Adil/G-4311-2010     2.444828           129   
8659                Karaca, Ezgi/P-1759-2019     2.430756            27   
15519         Topaloglu, Şeyda/AAB-3269-2021     2.400441            39   
14182              Selver, Alper/V-5118-2019     2.323138            42   
12230             Ozhan, Gunes/AAS-1681-2021     2.315628            

In [10]:
# Case-insensitive search
specific_researcher = top_matches[top_matches['Researcher'].str.contains('ABG-9392-2021', case=False, na=False)]

# Print the result
print(specific_researcher[['Researcher', 'final_score', 'num_articles', 'adjusted_relevance']])


                      Researcher  final_score  num_articles  \
7951  Işık, Zerrin/ABG-9392-2021     2.575826            22   

      adjusted_relevance  
7951            0.918341  
